# **PROBABILIDAD DE QUE UN ESTUDIANTE APRUEBE UNA ASIGNATURA**

In [2]:
# Instalamos la librería pgmpy para trabajar con redes bayesianas
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.0 MB/s eta 0:00:00


In [3]:
# Importamos las librerías de redes bayesianas
# para crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork
# para crear una tabla de probabilidades condicionales
from pgmpy.factors.discrete import TabularCPD
# para realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination


/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## Crear la estrutura de la red

In [4]:
# creamos la estructura de la red bayesiana

#La relación nos indica:
#HorasEstudio influye en Asistencia
#Asistencia influye en NotaExamen
#NotaExamen influye en Aprobación


model = DiscreteBayesianNetwork ([
    ("HorasEstudio", "Asistencia"),
    ("Asistencia", "NotaExamen"),
    ("NotaExamen", "Aprobacion")
])

## Probabilidad de HorasEstudio

In [7]:
# creamos la tabla de probabilidades de HorasEstudio
# es una variable raíz porque no depende de ninguna otra variable, esta solita

horas_estudio = TabularCPD(
    variable = "HorasEstudio",
    variable_card = 2,

# probabilidad de estudiar pocas o muchas horas

values = [
    [0.60],
    [0.40],
    ],
# nombres de los posibles estados de la variable

    state_names = {
        "HorasEstudio": ["Pocas", "Muchas"]
    }
)


## Probabilidad de Asistencia

In [11]:
# creamos la tabla de probabilidades de Asistencia
# y esta condicionada por las HorasEstudio

asistencia = TabularCPD(
    variable = "Asistencia",
    variable_card = 2,

values = [
    [0.80, 0.30],
    [0.20, 0.70],
],

# este indica que asistencia  depende de HorasEstudio

  evidence  = ["HorasEstudio"],
# Los dos estados de HorasEstudio
  evidence_card = [2],

#Nombramos los estados de las variables
    state_names = {
        "Asistencia": ["Baja", "Alta"],
        "HorasEstudio": ["Pocas", "Muchas"]
    }
)

## Probabilidad de NotaExamen

In [13]:
# creamo0s la tabla de probabilidades de NotaExamen
# y esta condicionada por el nivel de Asistencia

nota_examen = TabularCPD(
    variable = "NotaExamen",
    variable_card = 2,

values = [
    [0.70, 0.20],
    [0.30, 0.80],
],

# NotaExamen depende de Asistencia

evidence  = ["Asistencia"],
# Los dos estados de Asistencia
  evidence_card = [2],
# nombres de los estados
    state_names = {
        "NotaExamen": ["Baja", "Alta"],
        "Asistencia": ["Baja", "Alta"]
    }
)

## Probabilidad de Aprobación

In [14]:
# creamos la tabla de probabilides de Aprobación
# y esta condicionada por la NotaExamen

aprobacion = TabularCPD(
    variable = "Aprobacion",
    variable_card = 2,

values = [
    [0.95, 0.10],
    [0.05, 0.90],
],

# aprobacion depende de la NotaExamen
evidence = ["NotaExamen"],
#los dos estados de NotaExamen
    evidence_card = [2],
# nombres de los estados
    state_names = {
        "Aprobacion": ["NoAprobado", "Aprobado"],
        "NotaExamen": ["Baja", "Alta"]
    }
)

## Agregamos las probabilidades al modelo

In [16]:
#agregamos todas las tablas de probabilidades al modelo

model.add_cpds(
    horas_estudio,
    asistencia,
    nota_examen,
    aprobacion
)

#verificamos que la estructura de la red y todas sus probabilidades sean correctas
model.check_model()

True

## Mostrar quye la red haya sido creada

In [17]:
print("Modelo de red bayesiana creado y verificado correctamente.")

# Mostrar las relaciones entre las variables
print("Relaciones de la red:")
print(model.edges())

Modelo de red bayesiana creado y verificado correctamente.
Relaciones de la red:
[('HorasEstudio', 'Asistencia'), ('Asistencia', 'NotaExamen'), ('NotaExamen', 'Aprobacion')]


## Crear el motor de inferencia

In [18]:
# Creamos el motor de inferencia utilizando el modelo construido.
# VariableElimination permite calcular probabilidades
# teniendo o no teniendo información sobre algunas variables.

inferencia = VariableElimination(model)

## ¿Cuál es la probabilidad general de que un estudiante apruebe la asignatura?

In [19]:
# Consultamos la probabilidad de que un estudiante apruebe
# la asignatura sin proporcionar ninguna evidencia.
#
# Es decir, queremos conocer:
# P(Aprobacion = Aprueba)

resultado = inferencia.query(
    variables=["Aprobacion"]
)

# Mostrar el resultado de la consulta
print("Probabilidad de aprobación:")
print(resultado)

Probabilidad de aprobación:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(NoAprobado) |            0.5250 |
+------------------------+-------------------+
| Aprobacion(Aprobado)   |            0.4750 |
+------------------------+-------------------+


## Mostrar solamente el porcentaje de aprobación

In [20]:
# Obtenemos solo la probabilidad de que el estudiante apruebe
prob_aprobar = resultado.values[1]

# Convertimos la probabilidad a porcentaje
porcentaje_aprobar = prob_aprobar * 100

# Mostramos el resultado
print(f"Probabilidad de aprobar: {porcentaje_aprobar:.2f}%")

Probabilidad de aprobar: 47.50%


## Mostrar ambas probabilidades

In [21]:
# Obtenemos las probabilidades de cada estado de Aprobacion
prob_desaprobar = resultado.values[0]
prob_aprobar = resultado.values[1]

# q muestre las probabilidades en porcentaje
print("Resultados de la red bayesiana")
print("------------------------------")
print(f"Probabilidad de desaprobar: {prob_desaprobar * 100:.2f}%")
print(f"Probabilidad de aprobar:    {prob_aprobar * 100:.2f}%")

Resultados de la red bayesiana
------------------------------
Probabilidad de desaprobar: 52.50%
Probabilidad de aprobar:    47.50%


## Interpretación

Segun la red bayesiana que construí a partir de las variables y probabilidades del slide semana 5, la probabilidad general de que un estudiante apruebe la asignatura es de 47.50% mientras que la probabilidad de que un estudiante desapruebe es de 52.50%.. este resultado se obtuvo considerando las horas de estudio y la asistencia y la nota del examen